# Dazo v0 — ProofWriter recurrent-depth pilot

This notebook is **safe to rerun in the same Colab runtime**. It updates `/content/dazo` to the latest `8dazo/dazo` `main`, reuses any already-prepared ProofWriter pilot data, validates the architecture/losses, trains the recurrent head, and evaluates the same checkpoint at 1/2/4/6/8 recurrent steps.

**Runtime:** GPU. A Tesla T4 is sufficient for this pilot.


In [ ]:
!nvidia-smi || true
%cd /content

import os, shutil, subprocess
repo = '/content/dazo'
if os.path.isdir(os.path.join(repo, '.git')):
    subprocess.run(['git', '-C', repo, 'fetch', 'origin', 'main'], check=True)
    subprocess.run(['git', '-C', repo, 'reset', '--hard', 'origin/main'], check=True)
else:
    if os.path.exists(repo):
        shutil.rmtree(repo)
    subprocess.run(['git', 'clone', '-q', 'https://github.com/8dazo/dazo.git', repo], check=True)

%cd /content/dazo
!pip -q install -e ".[train]"
!git rev-parse --short HEAD


## 1. Architecture + loss invariants

This includes the regression for the CUDA-autocast BCE failure seen on the first T4 run.


In [ ]:
!pytest -q tests/test_core.py tests/test_losses.py


## 2. Prepare/reuse the ProofWriter pilot

Training keeps only examples with gold query depth ≤3. Validation/test retain deeper cases. If these files already exist in this Colab runtime, this cell reuses them instead of downloading/preparing them again.


In [ ]:
import os, subprocess
paths = [
    'data/proofwriter-pilot/train.jsonl',
    'data/proofwriter-pilot/validation.jsonl',
    'data/proofwriter-pilot/test.jsonl',
]
if all(os.path.exists(p) for p in paths):
    print('Reusing existing ProofWriter pilot data:')
    for p in paths:
        with open(p, 'r', encoding='utf-8') as f:
            print(p, sum(1 for _ in f), 'rows')
else:
    subprocess.run([
        'python', 'scripts/prepare_proofwriter.py',
        '--output', 'data/proofwriter-pilot',
        '--train-max-depth', '3',
        '--limit-train', '3000',
        '--limit-eval', '1000',
    ], check=True)


## 3. Train Dazo-0

The mmBERT-small backbone stays frozen. On a T4, Dazo automatically uses **FP16 + GradScaler**. Training samples recurrent budgets from 1/2/3/4/6/8 per batch.

This cell removes any incomplete pilot output from an earlier crashed run before training.


In [ ]:
!rm -rf outputs/dazo-proofwriter-pilot
!python train.py \
  --config configs/dazo-v0-small.json \
  --train data/proofwriter-pilot/train.jsonl \
  --eval data/proofwriter-pilot/validation.jsonl \
  --output outputs/dazo-proofwriter-pilot \
  --epochs 1 \
  --batch-size 4 \
  --grad-accum 2 \
  --lr 2e-4 \
  --depth-budgets 1,2,3,4,6,8


## 4. Verify the checkpoint

Do not continue to evaluation unless this cell succeeds.


In [ ]:
from pathlib import Path
ckpt = Path('outputs/dazo-proofwriter-pilot/final')
required = [ckpt/'config.json']
missing = [str(p) for p in required if not p.exists()]
assert not missing, f'Training did not finish; missing: {missing}'
print('Checkpoint ready:', ckpt.resolve())
print('Files:', sorted(p.name for p in ckpt.iterdir()))


## 5. Test-time compute scaling

This is the Gate-1 question: does the **same checkpoint** improve on deeper ProofWriter examples when given more recurrent inference steps? The report includes accuracy, Brier, ECE, depth slices, wrong→correct flips, correct→wrong flips, and overthinking rate.


In [ ]:
!python evaluate.py \
  --model outputs/dazo-proofwriter-pilot/final \
  --data data/proofwriter-pilot/test.jsonl \
  --batch-size 8 \
  --loops 1,2,4,6,8 \
  --output outputs/dazo-proofwriter-pilot/gate1-metrics.json


## 6. Inspect the Gate-1 report


In [ ]:
import json
from pathlib import Path
report_path = Path('outputs/dazo-proofwriter-pilot/gate1-metrics.json')
report = json.loads(report_path.read_text())
for loop, metrics in report['loops'].items():
    print(f"loops={loop:>2}  acc={metrics['accuracy']:.4f}  brier={metrics['brier']:.4f}  ece={metrics['ece']:.4f}  by_depth={metrics['by_depth']}")
print('\noverthinking_rate =', report.get('overthinking_rate'))
print('correctness_transitions =')
print(json.dumps(report.get('correctness_transitions', {}), indent=2))


## 7. What comes after the pilot

Do **not** interpret the 3k/1-epoch pilot as a final reasoning result. If the pipeline is stable, the next experiment is ~20k–50k shallow-depth examples for 2–3 epochs, plus a 1-loop/no-recurrence ablation. We only advance to latent-proof supervision or multi-branch reasoning if Gate 1 shows a reproducible positive compute-scaling curve.
